# Day 4 — Feature Engineering & Hyperparameter Tuning


Today I will focus on two ways of improving a Machine Learning workflow:

1. **Feature Engineering** — creating more useful information from the existing data.
2. **Hyperparameter Tuning** — systematically searching for better model settings.


### Main Questions

- Can engineered features provide more useful information than the original features?
- How does an untuned Random Forest perform?
- Which hyperparameter configuration performs best?
- Does tuning improve the baseline?
- Which engineered feature is more influential in the final model?
- How does the selected model perform on completely unseen test data?


<hr style="height: 5px; background-color: #b33c3c; border: none;">


## Learning Objectives

By the end of this notebook, I should be able to:

- Explain why feature engineering can improve Machine Learning performance.
- Create meaningful features from existing columns.
- Understand feature creation, binning, encoding, datetime extraction, and scaling.
- Distinguish learned parameters from hyperparameters.
- Define a hyperparameter search space.
- Use `GridSearchCV` with 5-fold cross-validation.
- Interpret `best_params_`, `best_score_`, and `best_estimator_`.
- Compare a tuned model against an untuned baseline.
- Use model results and feature importance to support conclusions.
- Explain when `RandomizedSearchCV` is useful.


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 1. Loading the Dataset

## Question

**What information is available in the customer dataset, and which variable are we trying to predict?**

Before creating features, I need to inspect the original data. This gives me a starting point and prevents me from engineering features blindly.


In [47]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

df = pd.read_csv("customers_ml_lab.csv")

df.head()


,CustomerID,Age,MonthlyCharges,ContractMonths,SupportCalls,InternetUsage,PaymentDelay,CustomerSatisfaction,Churn
0,3001,25,85,6,5,420,3,4,1
1,3002,44,90,24,2,300,1,8,0
2,3003,31,70,12,4,350,2,6,0
3,3004,52,65,36,1,120,0,9,0
4,3005,28,95,8,3,500,4,5,1


## What does the data tell me?

The first rows show the structure of the customer dataset.

The target variable is **`Churn`**, which indicates whether a customer leaves the service.

The other columns describe customer behavior and account characteristics, such as:

- age,
- monthly charges,
- contract duration,
- support calls,
- internet usage,
- payment delay,
- and customer satisfaction.

I will now inspect the complete structure before deciding which features can be engineered.


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 2. Understanding the Data Before Changing It

## Question

**Are there missing values, unexpected data types, or identifier columns that should not be used as predictive features?**

This question matters because feature engineering should begin with an understanding of the raw data.


In [48]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)


Shape: (40, 9)

Columns:
['CustomerID', 'Age', 'MonthlyCharges', 'ContractMonths', 'SupportCalls', 'InternetUsage', 'PaymentDelay', 'CustomerSatisfaction', 'Churn']

Missing values:
CustomerID              0
Age                     0
MonthlyCharges          0
ContractMonths          0
SupportCalls            0
InternetUsage           0
PaymentDelay            0
CustomerSatisfaction    0
Churn                   0
dtype: int64

Data types:
CustomerID              int64
Age                     int64
MonthlyCharges          int64
ContractMonths          int64
SupportCalls            int64
InternetUsage           int64
PaymentDelay            int64
CustomerSatisfaction    int64
Churn                   int64
dtype: object


## What does the data tell me?

The inspection tells me:

- how many observations and columns are available,
- which columns contain missing values,
- and which variables are numeric or categorical.

The `CustomerID` column is an identifier. It identifies a customer but does not represent customer behavior, so I will exclude it from the predictive features.

The next question is more important for today's topic:

**Can I create new information by combining meaningful existing variables?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 3. Feature Engineering

## Question

**Can relationships between existing columns give the model information that is not represented as directly in the raw columns?**

Feature engineering is the process of creating, transforming, or selecting features so that the model receives more useful information.

A useful engineered feature should have a reason behind it. I should be able to explain what information the new feature represents.


## Common Feature Engineering Techniques

| Technique | What it does | Example |
|---|---|---|
| Feature creation | Combines existing variables | `price_per_sqm = price / area` |
| Binning | Groups continuous values | `age → young / adult / senior` |
| One-hot encoding | Converts categories into numeric columns | `city → city_Nablus, city_Ramallah` |
| Datetime extraction | Extracts information from dates | `order_date → month, day_of_week` |
| Scaling | Places numeric features on comparable scales | `StandardScaler`, `MinMaxScaler` |

For this dataset, I will focus on **feature creation**, because the available numerical variables allow me to create meaningful ratios.


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 4. Creating the First Engineered Feature

## Question

**Does the relationship between monthly charges and support calls provide useful information?**

A customer who pays a certain monthly amount while making many support calls may represent a different situation from a customer paying the same amount with very few support calls.

I will create:

`charges_per_support_call`

This represents monthly charges relative to the number of support calls.

I add `1` to the denominator so that customers with zero support calls do not cause division by zero.


In [49]:
df["charges_per_support_call"] = (
    df["MonthlyCharges"] / (df["SupportCalls"] + 1)
)

df[
    [
        "MonthlyCharges",
        "SupportCalls",
        "charges_per_support_call"
    ]
].head()


,MonthlyCharges,SupportCalls,charges_per_support_call
0,85,5,14.166667
1,90,2,30.000000
2,70,4,14.000000
3,65,1,32.500000
4,95,3,23.750000


## What does the data tell me?

The new column is now calculated for every customer.

The feature does not replace `MonthlyCharges` or `SupportCalls`. Instead, it adds a new representation of the relationship between them.

### Next question

**Can another relationship in the dataset provide additional information?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 5. Creating the Second Engineered Feature

## Question

**Does internet usage relative to contract duration provide useful information?**

Two customers can have similar total internet usage but very different contract durations.

I will therefore create:

`usage_per_contract_month`

This represents internet usage relative to contract duration.


In [ ]:
df["usage_per_contract_month"] = (df["InternetUsage"] / (df["ContractMonths"] + 1))

df[["InternetUsage","ContractMonths","usage_per_contract_month"]].head()


,InternetUsage,ContractMonths,usage_per_contract_month
0,420,6,60.000000
1,300,24,12.000000
2,350,12,26.923077
3,120,36,3.243243
4,500,8,55.555556


## What does the data tell me?

The second engineered feature has been created successfully.

At this point, I have added two features based on relationships already present in the dataset:

- `charges_per_support_call`
- `usage_per_contract_month`

However, creating a feature does **not** automatically mean that it improves the model.

So the next question is:

**Do these features actually help predictive performance?**

To answer that, I first need a fair baseline.


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 6. Separating Features and Target

## Question

**What information should the model receive, and what value should it predict?**

`X` will contain the predictive features.

`y` will contain the target, `Churn`.

I will also remove `CustomerID` because it is an identifier rather than a meaningful predictive variable.


In [51]:
x = df.drop(columns=["Churn", "CustomerID"])
y = df["Churn"]

print("Feature shape:", x.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(x.columns.tolist())


Feature shape: (40, 9)
Target shape: (40,)

Features:
['Age', 'MonthlyCharges', 'ContractMonths', 'SupportCalls', 'InternetUsage', 'PaymentDelay', 'CustomerSatisfaction', 'charges_per_support_call', 'usage_per_contract_month']


## What does the data tell me?

The feature list now includes the two engineered variables.

The target `Churn` is kept separately, so the model cannot directly see the answer during training.

### Next question

**How should I separate the data so that I can develop the model without using the final test set for tuning?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 7. Creating Training and Test Data

## Question

**Can I keep a completely unseen test set for the final evaluation?**

Yes.

I will use 80% of the data for training and model development and keep 20% untouched for the final test.

The training portion will be used for cross-validation and hyperparameter tuning.

The test portion will not be used to choose the model.


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,stratify=y,random_state=42)

print("Training samples:", len(x_train))
print("Test samples:", len(x_test))


Training samples: 32
Test samples: 8


## What does the data tell me?

The data is now divided into:

- a training set for model development,
- an untouched test set for final evaluation.

`stratify=y` helps preserve the class distribution between the two sets.

### Next question

**Before tuning anything, how strong is the Random Forest baseline?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 8. Building the Untuned Baseline

## Question

**How well does a Random Forest perform before I deliberately search for better hyperparameters?**

I need a baseline because otherwise I cannot determine whether tuning actually helped.

I will evaluate the baseline using 5-fold Stratified Cross-Validation and F1 score.


In [ ]:
cv =5

model = RandomForestClassifier(random_state=42)

model_scores = cross_val_score(model,x_train,y_train,cv=cv,scoring="f1")

model_mean = model_scores.mean()
model_std = model_scores.std()

print("Baseline F1 scores:", model_scores)
print("Baseline mean F1:", model_mean)
print("Baseline std F1:", model_std)


Baseline F1 scores: [0.5        0.4        1.         0.85714286 0.66666667]
Baseline mean F1: 0.6847619047619047
Baseline std F1: 0.22119854924013638


## What does the data tell me?

The five F1 values show how the baseline behaves across different validation folds.

The **mean F1** represents the model's average cross-validated performance.

The **standard deviation** tells me how much the performance changes between folds.

I will use this baseline as the reference point for the tuning experiment.

### Next question

**Which settings of the Random Forest could be changed to search for better performance?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 9. Parameters vs. Hyperparameters

## Question

**What exactly am I changing when I tune a model?**

A **parameter** is learned by the model from training data.

A **hyperparameter** is a setting selected before or around the training process.

For Random Forest, examples include:

- `n_estimators` — number of trees.
- `max_depth` — maximum depth of each tree.
- `min_samples_split` — minimum samples required to split a node.

These are not learned in the same way as the model's internal parameters.

### Why search for them?

Choosing them manually can be slow and subjective.

Instead, I can define several possible values and let `GridSearchCV` compare them using cross-validation.


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 10. Defining the Hyperparameter Grid

## Question

**Which combinations of Random Forest settings should I test?**

I will test:

- 2 values for `n_estimators`,
- 3 values for `max_depth`,
- 2 values for `min_samples_split`.

This gives:

**2 × 3 × 2 = 12 combinations**

With 5-fold cross-validation:

**12 × 5 = 60 model fits.**


In [54]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

total_combinations = (
    len(param_grid["n_estimators"])
    * len(param_grid["max_depth"])
    * len(param_grid["min_samples_split"])
)

print("Parameter combinations:", total_combinations)
print("Total model fits:", total_combinations * 5)


Parameter combinations: 12
Total model fits: 60


## What does the data tell me?

The calculation shows exactly how expensive this search will be.

This is important because Grid Search grows quickly as more values are added.

### Next question

**Which of these 12 configurations gives the best cross-validated F1 score?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 11. Running GridSearchCV

## Question

**Which hyperparameter combination gives the strongest average F1 score?**

`GridSearchCV` will:

1. take every combination from the grid,
2. train the model using the training data,
3. evaluate each configuration using 5-fold cross-validation,
4. calculate the average F1 score,
5. select the best-performing configuration.


In [ ]:
grid = GridSearchCV(estimator=RandomForestClassifier(random_state=42),param_grid=param_grid,cv=cv,scoring="f1",n_jobs=-1,return_train_score=True)

grid.fit(x_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest cross-validated F1:")
print(grid.best_score_)


Best parameters:
{'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}

Best cross-validated F1:
0.7133333333333333


## What does the data tell me?

The output above gives the winning hyperparameter combination and its cross-validated F1 score.

`best_params_` tells me **which settings won**.

`best_score_` tells me **how well that winning configuration performed on average across the five folds**.

This is much more reliable than choosing a configuration after looking at only one validation split.

### Next question

**Was the winning configuration clearly better than the other configurations, or were several configurations similar?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 12. Examining All Grid Search Results

## Question

**How did the other tested configurations perform?**

I will inspect `cv_results_` so that the conclusion is based on all of the experiments rather than only the winner.


In [56]:
results = pd.DataFrame(grid.cv_results_)

results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")


,param_n_estimators,param_max_depth,param_min_samples_split,mean_test_score,std_test_score,rank_test_score
5,200,10,2,0.713333,0.249087,1
9,200,None,2,0.713333,0.249087,1
2,100,5,5,0.684762,0.221199,3
0,100,5,2,0.684762,0.221199,3
3,200,5,5,0.684762,0.221199,3
4,100,10,2,0.684762,0.221199,3
6,100,10,5,0.684762,0.221199,3
1,200,5,2,0.684762,0.221199,3
7,200,10,5,0.684762,0.221199,3
8,100,None,2,0.684762,0.221199,3


## What does the data tell me?

The results table shows the performance of every tested configuration.

- `mean_test_score` is the average F1 across the five folds.
- `std_test_score` shows variation across folds.
- `rank_test_score` identifies the ranking of each configuration.

The top-ranked row should match `grid.best_params_`.

This lets me support the tuning conclusion with the actual experiment rather than simply stating that tuning is useful.

### Next question

**Did the tuned model actually improve over the baseline?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 13. Baseline vs. Tuned Model

## Question

**Did systematic hyperparameter tuning improve the model?**

The fairest comparison is between:

- the baseline mean cross-validated F1,
- the best mean cross-validated F1 found by GridSearchCV.


In [57]:
tuned_mean = grid.best_score_
improvement = tuned_mean - model_mean

comparison = pd.DataFrame({
    "Model": [
        "Untuned Baseline",
        "Tuned Random Forest"
    ],
    "Mean CV F1": [
        model_mean,
        tuned_mean
    ]
})

print(comparison)
print("\nF1 improvement:", improvement)


                 Model  Mean CV F1
0     Untuned Baseline    0.684762
1  Tuned Random Forest    0.713333

F1 improvement: 0.02857142857142858


## What does the data tell me?

The value of `improvement` answers the question directly.

- If it is **positive**, the tuned configuration improved the average CV F1.
- If it is **close to zero**, tuning produced little practical change.
- If it is **negative**, the tested search space did not beat the baseline.

The important conclusion must come from the number produced by this experiment.

### Next question

**Which hyperparameter values were associated with stronger performance?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 14. Investigating Hyperparameter Influence

## Question

**Which hyperparameter appears to matter most within the values I tested?**

I will group the Grid Search results by each hyperparameter and calculate the average F1 for its tested values.

This does not prove that one hyperparameter is universally the most important. It only describes the behavior observed in this experiment.


In [58]:
for parameter in [
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split"
]:
    summary = (
        results.groupby(parameter)["mean_test_score"]
        .mean()
        .sort_values(ascending=False)
    )

    print(f"\nAverage F1 by {parameter}:")
    print(summary)



Average F1 by param_n_estimators:
param_n_estimators
200    0.694286
100    0.684762
Name: mean_test_score, dtype: float64

Average F1 by param_max_depth:
param_max_depth
10    0.691905
5     0.684762
Name: mean_test_score, dtype: float64

Average F1 by param_min_samples_split:
param_min_samples_split
2    0.694286
5    0.684762
Name: mean_test_score, dtype: float64


## What does the data tell me?

For each hyperparameter, the table shows how its tested values performed on average across the other configurations.

The value with the highest average F1 is the strongest candidate **within this search**.

This gives me a data-based answer instead of guessing which Random Forest hyperparameter matters most.

### Next question

**Which of my engineered features was more influential in the final Random Forest?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 15. Investigating the Engineered Features

## Question

**Did one of the engineered features contribute more to the final model than the other?**

The tuned Random Forest provides feature importance values.

I will inspect the two engineered features specifically.


In [59]:
best_model = grid.best_estimator_

feature_importance = pd.Series(
    best_model.feature_importances_,
    index=x_train.columns
).sort_values(ascending=False)

print("All feature importances:")
print(feature_importance)

print("\nEngineered features:")
print(
    feature_importance[
        [
            "charges_per_support_call",
            "usage_per_contract_month"
        ]
    ]
)


All feature importances:
MonthlyCharges              0.179811
ContractMonths              0.152835
usage_per_contract_month    0.144206
CustomerSatisfaction        0.108368
charges_per_support_call    0.107841
InternetUsage               0.104793
SupportCalls                0.078192
Age                         0.076748
PaymentDelay                0.047206
dtype: float64

Engineered features:
charges_per_support_call    0.107841
usage_per_contract_month    0.144206
dtype: float64


## What does the data tell me?

The feature importance values allow me to compare the two engineered features inside the fitted Random Forest.

The feature with the larger importance had a greater contribution to the decisions made by this particular fitted model.

I should phrase the conclusion carefully:

> **The data shows which engineered feature was more influential in this Random Forest experiment. It does not prove that the feature is universally more important for every model or dataset.**

### Next question

**Does the selected model still perform well when it finally sees completely unseen data?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 16. Final Evaluation on the Untouched Test Set

## Question

**How well does the final selected model generalize to data that was not used during model selection?**

The test set has been kept untouched.

Now I can train the selected configuration on the full training data and evaluate it once on the test set.


In [60]:
best_model.fit(x_train, y_train)

test_predictions = best_model.predict(x_test)

test_f1 = f1_score(y_test, test_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print("Final Test F1:", test_f1)
print("Final Test Accuracy:", test_accuracy)


Final Test F1: 1.0
Final Test Accuracy: 1.0


## What does the data tell me?

The final test scores show how the selected model performs on unseen customer records.

The important distinction is:

- `grid.best_score_` = cross-validated score used during model selection.
- `test_f1` = final score on unseen test data.

The test result should be reported as a final evaluation, not used to decide which hyperparameters to try next.

### Final question

**What did this entire experiment teach me about feature engineering and hyperparameter tuning?**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 17. Final Data-Driven Conclusion

## Question

**What can I conclude from the actual experiments in this notebook?**

I should answer this using the values produced above rather than making a general claim that tuning or feature engineering always works.

### Experiment 1 — Feature Engineering

I created:

- `charges_per_support_call`
- `usage_per_contract_month`

These features represented relationships between existing customer variables.

### Experiment 2 — Baseline

The untuned Random Forest produced:

**Mean CV F1 = `baseline_mean`**

**Std CV F1 = `baseline_std`**

### Experiment 3 — Hyperparameter Tuning

GridSearchCV tested:

**12 configurations × 5 folds = 60 fits**

The best configuration was:

**`grid.best_params_`**

with:

**Best CV F1 = `grid.best_score_`**

### Experiment 4 — Comparison

The improvement over the baseline was:

**`improvement`**

Therefore, the conclusion about whether tuning helped should be based on the sign and size of this value.

### Experiment 5 — Engineered Features

The feature importance output showed which of the two engineered features was more influential in the fitted Random Forest.

### Experiment 6 — Final Generalization

The final untouched test results were:

**Test F1 = `test_f1`**

**Test Accuracy = `test_accuracy`**

The overall lesson is:

> **Feature engineering changes the information available to the model, while hyperparameter tuning changes how the model is configured. Both should be evaluated experimentally rather than assumed to improve performance.**


<hr style="height: 5px; background-color: #b33c3c; border: none;">


# 18. Training Notebook — Final Takeaway

## Question

**What should I remember from today's lesson before moving to the Hands-On Lab?**

Today's workflow showed two different ways of improving a Machine Learning model:

### Feature Engineering

I changed the **representation of the information** given to the model.

For this training example, I created:

- `charges_per_support_call`
- `usage_per_contract_month`

The important lesson is that engineered features should have a reason behind them. Creating more columns is not automatically better.

### Hyperparameter Tuning

I changed the **configuration of the model**.

I used `GridSearchCV` to systematically test different Random Forest settings using 5-fold cross-validation.

The important outputs were:

- `best_params_`
- `best_score_`
- `best_estimator_`



<hr style="height: 5px; background-color: #b33c3c; border: none;">


# Tools Used

- **Pandas** — loading, inspecting, and transforming data
- **Scikit-learn** — Random Forest, cross-validation, GridSearchCV and evaluation metrics
- **Matplotlib** — optional visualization
- **Jupyter Notebook** — interactive analysis and narrative documentation


<hr style="height: 5px; background-color: #b33c3c; border: none;">
